# Training and evaluating Llama-3.2-3B

In [1]:
import os
import math
from huggingface_hub import login
import torch
import transformers
import time
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed
from datasets import Dataset
import matplotlib.pyplot as plt
import numpy as np
from peft import LoraConfig, PeftModel
from datetime import datetime
import json
from trl import SFTTrainer, SFTConfig
import pandas as pd

colab = False

if colab:
    from google.colab import drive
    drive.mount('/content/drive')
    data_path = Path("/content/drive/MyDrive/Colab Notebooks/3H1L/data")
    models_path = Path("/content/drive/MyDrive/Colab Notebooks/3H1L/models/Llama")
else:
    data_path = "../data"
    models_path = "../models/Llama"

with open(f"{data_path}/train_nn_set.json", "r") as f:
    train_nn_set = json.load(f)

with open(f"{data_path}/validate_set.json", "r") as f:
    validate_set = json.load(f)

with open(f"{data_path}/test_set.json", "r") as f:
    test_set = json.load(f)

f:\Mike\Documents\Backup\Python\fake-headline-predictor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Transform data into appropriate format

In [2]:
def prompt_transform(target_set, test):

    modified_set = []

    for item in target_set:
        headlines = "\n\n".join([f"{i + 1}: {headline}" for i, headline in enumerate(item["headlines"])])
        headlines = f"Identify which of the four following headlines is fake:\n\n{headlines}\n\nAnswer: "
        if test:
            entry = {
                "date": item["date"],
                "prompt": headlines,
                "answer": item["answer"],
            }
        else:
            entry = {
                "date": item["date"],
                "prompt": headlines,
                "answer": item["answer"],
                "completion": str(item["answer"] + 1)
            }

        modified_set.append(entry)

    return modified_set

train_set_Llama_ready = prompt_transform(train_nn_set, False)
validate_set_Llama_ready = prompt_transform(validate_set, False)
test_set_Llama_ready = prompt_transform(test_set, True)

In [3]:
train_dataset = Dataset.from_list(train_set_Llama_ready)
validate_dataset = Dataset.from_list(validate_set_Llama_ready)
test_dataset = Dataset.from_list(test_set_Llama_ready)

n_train = len(train_dataset)
n_validate = len(validate_dataset)
n_test = len(test_dataset)

### Model config

Quantisation

In [4]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
SEED = 42
set_seed(SEED)
torch_dtype=torch.float16,

if colab:
    capability = torch.cuda.get_device_capability()
    use_bf16 = capability[0] >= 8
else:
    use_bf16 = False

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )

In [5]:
# Load the Tokenizer and the Model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 254/254 [00:38<00:00,  6.61it/s]


Memory footprint: 2197.6 MB


In [6]:
MAX_SEQUENCE_LENGTH = 256

for name, dataset in [
    ("Train", train_dataset),
    ("Validation", validate_dataset),
    ("Test", test_dataset),
]:
    lengths = [
        len(tokenizer(item["prompt"], add_special_tokens=True)["input_ids"])
        for item in dataset
    ]

    print(
        f"{name}: max={max(lengths)}, "
        f"mean={sum(lengths) / len(lengths):.1f}, "
        f"over {MAX_SEQUENCE_LENGTH}={sum(length > MAX_SEQUENCE_LENGTH for length in lengths)}"
    )

Train: max=117, mean=87.6, over 256=0
Validation: max=132, mean=87.5, over 256=0
Test: max=112, mean=87.1, over 256=0


Parameter definition, assignment and storage and model training (only run in Colab)

In [17]:
set_seed(SEED)

# Define metadata saving functions
def create_run_dir(base=models_path):
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    run_dir = Path(base) / timestamp
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir

# Create run directories
run_dir = create_run_dir()

# LoRA parameters

LORA_R = 32
LORA_ALPHA = LORA_R * 2
LORA_DROPOUT = 0.1

ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
TARGET_MODULES = ATTENTION_LAYERS

LORA_BIAS = "none"
LORA_TASK_TYPE = "CAUSAL_LM"


# Training parameters

EPOCHS = 3
BATCH_SIZE = 8
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_32bit"

WARMUP_STEPS = 1
LR_SCHEDULER_TYPE = "cosine"

MAX_GRAD_NORM = 0.3
MAX_SEQUENCE_LENGTH = 256

SAVE_STRATEGY = "epoch"
SAVE_TOTAL_LIMIT = 3

EVAL_STRATEGY = "epoch"

LOGGING_STEPS = 1
REPORT_TO = "none"

LOAD_BEST_MODEL_AT_END = True
METRIC_FOR_BEST_MODEL = "eval_loss"
GREATER_IS_BETTER = False

GROUP_BY_LENGTH = True

# Assign LoRA configuration

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias=LORA_BIAS,
    task_type=LORA_TASK_TYPE,
    target_modules=TARGET_MODULES,
)

# Assign training args

output_dir = str(run_dir / "checkpoints")

train_parameters = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_strategy=SAVE_STRATEGY,
    save_total_limit=SAVE_TOTAL_LIMIT,
    logging_steps=LOGGING_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=MAX_GRAD_NORM,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to=REPORT_TO,
    max_length=MAX_SEQUENCE_LENGTH,
    eval_strategy=EVAL_STRATEGY,
    load_best_model_at_end=LOAD_BEST_MODEL_AT_END,
    metric_for_best_model=METRIC_FOR_BEST_MODEL,
    greater_is_better=GREATER_IS_BETTER,
    seed=SEED
)

trainer = SFTTrainer(
    model=base_model,
    train_dataset=train_dataset,
    eval_dataset=validate_dataset,
    peft_config=lora_parameters,
    args=train_parameters,
    processing_class=tokenizer,
)

# TRL converts QLoRA trainable parameters to BF16; T4 requires FP16
for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

num_model_params = sum(p.numel() for p in trainer.model.parameters())
trainable_params = sum(
    p.numel() for p in trainer.model.parameters()
    if p.requires_grad
)

print(Counter(
    str(p.dtype) for p in trainer.model.parameters()
    if p.requires_grad
))

output = trainer.train()

run_metadata = {
    "Model Information": {
        "Model name": BASE_MODEL,
        "Number of model parameters": num_model_params,
        "Number of trainable parameters": trainable_params,
        "Trainable percentage": 100 * trainable_params / num_model_params
    },
    "Dataset Information": {
        "Training set size": n_train,
        "Validation set size": n_validate,
        "Test set size": n_test,
    },
    "Tokenizer Settings": {
        "Padding setting": tokenizer.padding_side,
        "Max length": MAX_SEQUENCE_LENGTH,
    },
    "Quantisation Settings": {
        "Load in 4-bit": quant_config.load_in_4bit,
        "Double quantisation": quant_config.bnb_4bit_use_double_quant,
        "Quantisation type": quant_config.bnb_4bit_quant_type,
        "Compute dtype": str(quant_config.bnb_4bit_compute_dtype),
    },
    "LoRA Settings": {
        "Rank": LORA_R,
        "Alpha": LORA_ALPHA,
        "Dropout": LORA_DROPOUT,
        "Bias": LORA_BIAS,
        "Task type": LORA_TASK_TYPE,
        "Target modules": TARGET_MODULES,
    },
    "Training Hyperparameters": {
        "Number of training epochs": EPOCHS,
        "Per-device training batch size": BATCH_SIZE,
        "Per-device eval batch size": PER_DEVICE_EVAL_BATCH_SIZE,
        "Gradient accumulation steps": GRADIENT_ACCUMULATION_STEPS,
        "Output directory": output_dir,
        "Save strategy": SAVE_STRATEGY,
        "Save total limit": SAVE_TOTAL_LIMIT,
        "Logging steps": LOGGING_STEPS,
        "Report to": REPORT_TO,
        "Learning rate": LEARNING_RATE,
        "Optimiser": OPTIMIZER,
        "Weight decay": WEIGHT_DECAY,
        "Warmup steps": WARMUP_STEPS,
        "Learning rate scheduler type": LR_SCHEDULER_TYPE,
        "Maximum gradient norm": MAX_GRAD_NORM,
        "Precision": "bfloat16" if use_bf16 else "float16",
        "Seed": SEED,
        "Load best model at end": LOAD_BEST_MODEL_AT_END,
        "Metric for best model": METRIC_FOR_BEST_MODEL,
        "Greater is better": GREATER_IS_BETTER,
    },
    "Training Results": {
        "Training time and speed": output.metrics,
        "Best model checkpoint": trainer.state.best_model_checkpoint,
        "Best validation loss": trainer.state.best_metric,
    }
}

def convert_numpy(obj):
    if isinstance(obj, dict):
        return {k: convert_numpy(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy(v) for v in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    elif isinstance(obj, (np.floating, float)):
        return round(float(obj), 6)
    else:
        return obj

with open(f"{run_dir}/run_metadata.json", "w") as f:
    json.dump(convert_numpy(run_metadata), f, indent=4)

print(f"Run metadata saved to {run_dir}/run_metadata.json")

# Extract log history
log_history = trainer.state.log_history

# Training loss
train_points = [x for x in log_history if "loss" in x]
train_steps = [p["step"] for p in train_points]
train_losses = [p["loss"] for p in train_points]

# Validation loss
val_points = [x for x in log_history if "eval_loss" in x]
val_steps = [p["step"] for p in val_points]
val_losses = [p["eval_loss"] for p in val_points]

# Plot
plt.figure(figsize=(8,5))
plt.plot(train_steps, train_losses, label="Training Loss")
plt.plot(val_steps, val_losses, label="Validation Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.savefig(run_dir / "loss_curve.png")
plt.close()


/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/203 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/203 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/203 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/203 [00:00<?, ? examples/s]

...

Building labels for eval dataset:   0%|          | 0/67 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/67 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/67 [00:00<?, ? examples/s]

Counter({'torch.float32': 224})


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.711853,0.548644,0.597275,18193.000000,0.761194
2,0.880267,0.533002,0.495866,36386.000000,0.761194
3,0.066615,0.634734,0.304111,54579.000000,0.753731


Logging and storing (only run in Colab)

In [18]:
log_history = trainer.state.log_history

clean_log = []
last_train_loss = None
last_step = None

# Record values at each logging step
for entry in log_history:
    step = entry.get("step")
    if "loss" in entry:
        last_train_loss = entry["loss"]
    if "eval_loss" in entry:
        # Skip if this step is already appended
        if step == last_step:
            continue
        clean_entry = {
            "step": step,
            "train_loss": last_train_loss,
            "eval_loss": entry.get("eval_loss"),
            "epoch": entry.get("epoch")
        }
        clean_log.append(clean_entry)
        last_step = step

# Save JSON
with open(run_dir / "log_history.json", "w") as f:
    json.dump(clean_log, f, indent=2)

# Save CSV
df = pd.DataFrame(clean_log)
df.to_csv(run_dir / "log_history.csv", index=False)

# Save best model
best_model_dir = run_dir / "best_model"
trainer.save_model(best_model_dir)
tokenizer.save_pretrained(best_model_dir)

('/content/drive/MyDrive/Colab Notebooks/3H1L/models/Llama/20261005-143410/best_model/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/3H1L/models/Llama/20261005-143410/best_model/tokenizer.json')

Load model for evaluation

In [13]:
test_dataset[0]

{'date': '2019-11-23',
 'prompt': 'Identify which of the four following headlines is fake:\n\n1: Taylor Swift fans boycott J.Crew over private equity ties\n\n2: Cambridge University students cry fowl over 17th century painting that upsets vegetarians\n\n3: Why the world is running out of sand\n\n4: Manager drives from New York to Pennsylvania to return keys\n\nAnswer: ',
 'answer': 0}

In [7]:
model_path = Path(f"{models_path}/20261005-143410/best_model")

model = AutoModelForCausalLM.from_pretrained(model_path)
model.eval()

test_inputs = {
    key: torch.tensor(test_dataset[key])
    for key in ["input_ids", "attention_mask"]
}

with torch.no_grad():
    outputs = model(**test_inputs)

logits = outputs.logits

Loading weights: 100%|██████████| 224/224 [00:01<00:00, 181.05it/s]


ValueError: Column 'input_ids' doesn't exist.

In [14]:
test_dataset.column_names

['date', 'prompt', 'answer']